# 1. A map is a graph

Everything in pathrun rests on one idea. For routing purposes a map is not a picture, it is a graph.

A junction is a **node**. The stretch of road or path between two junctions is an **edge**. The drawn
shape of that stretch is stored on the edge and used only for display. The router never looks at it.

This notebook takes the real network around home and pulls it apart until that claim is obvious.
Run `scripts/fetch_network.py` first, so the graph is cached.

In [ ]:
import matplotlib.pyplot as plt
import networkx
import osmnx

from pathrun import config, geocoding, network

HOME_LATITUDE, HOME_LONGITUDE = geocoding.home_coordinates()
graph = network.load_network(*geocoding.home_coordinates(), config.NETWORK_RADIUS_METRES)
type(graph), graph.number_of_nodes(), graph.number_of_edges()

## What kind of graph is it

It is a `MultiDiGraph`. Three things are packed into that name.

**Graph** means nodes joined by edges. **Di** means directed, so an edge runs from one node to
another and not automatically back. A two way street appears as two edges pointing opposite ways,
which is what lets one way streets be represented at all. **Multi** means two nodes can be joined by
more than one edge at once, which happens with a dual carriageway or a road and a parallel cycle
path sharing both endpoints.

That last part is why edges are addressed by three things and not two. You need the source, the
target, and a key to say which of the parallel edges you mean.

In [ ]:
# A node is just an id with coordinates hanging off it. Nothing about roads.
first_node = next(iter(graph.nodes))
graph.nodes[first_node]

In [ ]:
# An edge carries everything that describes the way itself.
source, target, key, attributes = next(iter(graph.edges(keys=True, data=True)))
print(f'edge {source} -> {target}, key {key}')
attributes

## One junction, by hand

Pick the node nearest home and look at what leaves it. The number of edges leaving a node is its
**out degree**. A dead end has one, a crossroads has four.

This is the entire local picture a routing algorithm ever gets. Standing at a node it can see the
edges leaving it and what each one costs. It cannot see the map.

In [ ]:
home_node = osmnx.distance.nearest_nodes(graph, X=HOME_LONGITUDE, Y=HOME_LATITUDE)
print(f'nearest node to home: {home_node}, out degree {graph.out_degree(home_node)}\n')
for _, neighbour, attributes in graph.out_edges(home_node, data=True):
    highway = attributes.get('highway')
    designation = attributes.get('designation', '-')
    print(f'  -> {neighbour}  {attributes["length"]:6.0f} m  {str(highway):<14} {designation}')

## What simplification removed

When the graph loaded, `simplify=True` did something worth understanding. A road is originally drawn
in OpenStreetMap with many points along it to capture its curve. Almost all of those points are not
junctions. Nothing decides anything there, so the router does not need them.

Simplification deletes those points as nodes and keeps them as a `geometry` attribute on the edge.
So the graph shrinks a lot while the drawn shape survives intact.

Find an edge with a curve and count the difference.

In [ ]:
curviest = max(graph.edges(data=True), key=lambda edge: len(edge[2]['geometry'].coords) if 'geometry' in edge[2] else 0)
source, target, attributes = curviest
shape_points = len(attributes['geometry'].coords)
print(f'this single edge is drawn with {shape_points} points, but the graph stores only its 2 end nodes')
print(f'so {shape_points - 2} points were removed as nodes and kept as geometry')

In [ ]:
# Why keeping the geometry matters: joining the two end nodes with a straight line cuts the corner.
line = attributes['geometry']
plt.figure(figsize=(6, 6))
plt.plot(*line.xy, label='real shape, from the edge geometry')
plt.plot([line.coords[0][0], line.coords[-1][0]], [line.coords[0][1], line.coords[-1][1]], '--', label='node to node straight line')
plt.legend(); plt.axis('equal'); plt.title('An exported GPX must use the geometry'); plt.show()

## Degree distribution

Most nodes in a simplified network have degree 1 or 3. Degree 2 should be rare, because a node with
one way in and one way out is exactly what simplification removes. Seeing few of them is the check
that simplification did its job.

The degree 2 nodes that survive are ones where something about the way changed at that point, such
as the surface or the name, so the two sides could not be merged into one edge.

In [ ]:
degrees = [degree for _, degree in graph.degree()]
plt.figure(figsize=(7, 4))
plt.hist(degrees, bins=range(1, 12), align='left', rwidth=0.8)
plt.xlabel('node degree'); plt.ylabel('nodes'); plt.title('Most junctions are dead ends or T junctions'); plt.show()

## Seeing a small piece of it

Plot the network within a kilometre of home. This is the last time the map looks like a map. From
here on it is nodes, edges and costs.

In [ ]:
local = osmnx.truncate.truncate_graph_dist(graph, home_node, 1000)
right_of_way_colour = ['tab:green' if network.is_right_of_way(attributes) else 'tab:grey'
                       for _, _, attributes in local.edges(data=True)]
osmnx.plot.plot_graph(local, edge_color=right_of_way_colour, node_size=6, edge_linewidth=1.2,
                      bgcolor='white', node_color='black', figsize=(9, 9))

Green is a designated public right of way, grey is everything else. That green is the whole reason
this project exists, and the next notebook starts asking how to find a path through it.